In [1]:
from astropy.io import fits
from astropy import units as u
from spectral_cube import SpectralCube as sc
import warnings
from astropy.utils.exceptions import AstropyWarning
from tqdm.notebook import trange
from astropy.convolution import Gaussian1DKernel
import numpy as np
from radio_beam import Beam

# Suppress the specific Astropy warning
warnings.filterwarnings('ignore', message='No observer defined on WCS', category=AstropyWarning)

%matplotlib widget

In [5]:
# Define the source and target cube patterns
source_pattern = "./South/Baseline Corrected/CRAFTS_RA{ra}_DEC-13_2_cor_{sign}.fits"
target_pattern = "../HIPASS/HIPASS_mosaic_cube_{ra}_{sign}.fits"
output_pattern = "./South/Smoothed/CRAFTS_RA{ra}_DEC-13_2_cor_smo_{sign}.fits"
# Define RA ranges and signs
ra_ranges = ["60_80", "80_100", "100_120", "120_140"]
signs = ["+", "-"]

# Enqueue all tasks
for i in trange(len(ra_ranges)):
    for j in trange(len(signs)):
        ra = ra_ranges[i]
        sign = signs[j]

        source_file = source_pattern.format(ra=ra, sign=sign)
        target_file = target_pattern.format(ra=ra, sign=sign)
        output_file = output_pattern.format(ra=ra, sign=sign)

        # Smooth the CRAFTS cube to match the HIPASS data
        cube = sc.read(source_file)
        cube = cube.with_beam(
            Beam(major=4 * u.arcmin, minor=4 * u.arcmin, pa=0 * u.deg)
        )
        cube.allow_huge_operations = True
        vmin = np.min(cube.spectral_axis)
        vmax = np.max(cube.spectral_axis)
        print("original cube", cube)

        with fits.open(target_file) as hdul:
            target_header = hdul[0].header
            hdul.close()

        target_beam = Beam.from_fits_header(target_header)
        target_beam

        cube_spatial_smoothed = cube.convolve_to(target_beam)
        print("after spatial smoothed", cube_spatial_smoothed)

        fwhm_factor = np.sqrt(8 * np.log(2))
        current_resolution = cube.header["CDELT3"] * u.m / u.s
        target_resolution = target_header["CDELT3"] * u.m / u.s
        pixel_scale = np.abs(current_resolution)
        gaussian_width = (
            (target_resolution**2 - current_resolution**2) ** 0.5
            / pixel_scale
            / fwhm_factor
        )
        kernel = Gaussian1DKernel(gaussian_width.value)

        # cube_spatial_spectral_smoothed = cube_spatial_smoothed.spectral_smooth(kernel)
        # print("after spectral smoothed", cube_spatial_spectral_smoothed)

        cube_spatial_spectral_smoothed = cube_spatial_smoothed.reproject(
            target_header, roundtrip_coords=False, parallel=False
        )
        print("after reprojected", cube_spatial_spectral_smoothed)
        cube_spatial_spectral_smoothed = cube_spatial_spectral_smoothed.subcube(zlo=vmin, zhi=vmax)
        print("after velocity cropped", cube_spatial_spectral_smoothed)

        cube_spatial_spectral_smoothed.write(output_file, overwrite=True)

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

original cube SpectralCube with shape=(994, 600, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:    60.012500 deg:   79.987500 deg
 n_y:    600  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    1.987500 deg
 n_s:    994  type_s: VRAD      unit_s: m / s  range:    83058.065 m / s:  282932.675 m / s
after spatial smoothed SpectralCube with shape=(994, 600, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:    60.012500 deg:   79.987500 deg
 n_y:    600  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    1.987500 deg
 n_s:    994  type_s: VRAD      unit_s: m / s  range:    83058.065 m / s:  282932.675 m / s
after reprojected SpectralCube with shape=(16, 226, 301) and unit=K:
 n_x:    301  type_x: RA---CAR  unit_x: deg    range:    60.020823 deg:   80.020824 deg
 n_y:    226  type_y: DEC--CAR  unit_y: deg    range:   -13.022828 deg:    1.977172 deg
 n_s:     16  type_s: VRAD      unit_s: m / s  range:    85640.152 m / s:  283

after spatial smoothed SpectralCube with shape=(995, 600, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:    60.012500 deg:   79.987500 deg
 n_y:    600  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    1.987500 deg
 n_s:    995  type_s: VRAD      unit_s: m / s  range:  -255098.374 m / s:  -55022.481 m / s
after reprojected SpectralCube with shape=(16, 226, 301) and unit=K:
 n_x:    301  type_x: RA---CAR  unit_x: deg    range:    60.020823 deg:   80.020824 deg
 n_y:    226  type_y: DEC--CAR  unit_y: deg    range:   -13.022828 deg:    1.977172 deg
 n_s:     16  type_s: VRAD      unit_s: m / s  range:  -257334.201 m / s:  -59464.382 m / s
after velocity cropped SpectralCube with shape=(16, 226, 301) and unit=K:
 n_x:    301  type_x: RA---CAR  unit_x: deg    range:    60.020823 deg:   80.020824 deg
 n_y:    226  type_y: DEC--CAR  unit_y: deg    range:   -13.022828 deg:    1.977172 deg
 n_s:     16  type_s: VRAD      unit_s: m / s  range:  -257334.201 m /

  0%|          | 0/2 [00:00<?, ?it/s]

...

original cube SpectralCube with shape=(995, 600, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:   100.012500 deg:  119.987500 deg
 n_y:    600  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    1.987500 deg
 n_s:    995  type_s: VRAD      unit_s: m / s  range:   131969.979 m / s:  332045.872 m / s
after spatial smoothed SpectralCube with shape=(995, 600, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:   100.012500 deg:  119.987500 deg
 n_y:    600  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    1.987500 deg
 n_s:    995  type_s: VRAD      unit_s: m / s  range:   131969.979 m / s:  332045.872 m / s
after reprojected SpectralCube with shape=(16, 226, 301) and unit=K:
 n_x:    301  type_x: RA---CAR  unit_x: deg    range:   100.020825 deg:  120.020826 deg
 n_y:    226  type_y: DEC--CAR  unit_y: deg    range:   -13.022828 deg:    1.977172 deg
 n_s:     16  type_s: VRAD      unit_s: m / s  range:   138405.437 m / s:  336

after spatial smoothed SpectralCube with shape=(994, 600, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:   100.012500 deg:  119.987500 deg
 n_y:    600  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    1.987500 deg
 n_s:    994  type_s: VRAD      unit_s: m / s  range:  -259929.180 m / s:  -60054.571 m / s
after reprojected SpectralCube with shape=(16, 226, 301) and unit=K:
 n_x:    301  type_x: RA---CAR  unit_x: deg    range:   100.020825 deg:  120.020826 deg
 n_y:    226  type_y: DEC--CAR  unit_y: deg    range:   -13.022828 deg:    1.977172 deg
 n_s:     16  type_s: VRAD      unit_s: m / s  range:  -257334.201 m / s:  -59464.382 m / s
after velocity cropped SpectralCube with shape=(16, 226, 301) and unit=K:
 n_x:    301  type_x: RA---CAR  unit_x: deg    range:   100.020825 deg:  120.020826 deg
 n_y:    226  type_y: DEC--CAR  unit_y: deg    range:   -13.022828 deg:    1.977172 deg
 n_s:     16  type_s: VRAD      unit_s: m / s  range:  -257334.201 m /

  0%|          | 0/2 [00:00<?, ?it/s]

original cube SpectralCube with shape=(995, 600, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:   120.012500 deg:  139.987500 deg
 n_y:    600  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    1.987500 deg
 n_s:    995  type_s: VRAD      unit_s: m / s  range:   139014.904 m / s:  339090.798 m / s
after spatial smoothed SpectralCube with shape=(995, 600, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:   120.012500 deg:  139.987500 deg
 n_y:    600  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    1.987500 deg
 n_s:    995  type_s: VRAD      unit_s: m / s  range:   139014.904 m / s:  339090.798 m / s
after reprojected SpectralCube with shape=(16, 226, 301) and unit=K:
 n_x:    301  type_x: RA---CAR  unit_x: deg    range:   120.020826 deg:  140.020827 deg
 n_y:    226  type_y: DEC--CAR  unit_y: deg    range:   -13.022828 deg:    1.977172 deg
 n_s:     16  type_s: VRAD      unit_s: m / s  range:   138405.437 m / s:  336

after spatial smoothed SpectralCube with shape=(994, 600, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:   120.012500 deg:  139.987500 deg
 n_y:    600  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    1.987500 deg
 n_s:    994  type_s: VRAD      unit_s: m / s  range:  -260935.598 m / s:  -61060.989 m / s
after reprojected SpectralCube with shape=(16, 226, 301) and unit=K:
 n_x:    301  type_x: RA---CAR  unit_x: deg    range:   120.020826 deg:  140.020827 deg
 n_y:    226  type_y: DEC--CAR  unit_y: deg    range:   -13.022828 deg:    1.977172 deg
 n_s:     16  type_s: VRAD      unit_s: m / s  range:  -257334.201 m / s:  -59464.382 m / s
after velocity cropped SpectralCube with shape=(16, 226, 301) and unit=K:
 n_x:    301  type_x: RA---CAR  unit_x: deg    range:   120.020826 deg:  140.020827 deg
 n_y:    226  type_y: DEC--CAR  unit_y: deg    range:   -13.022828 deg:    1.977172 deg
 n_s:     16  type_s: VRAD      unit_s: m / s  range:  -257334.201 m /